In [1]:
%cd ~/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
%load_ext autoreload
%autoreload 2
import sys
print(sys.executable)

/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室/.venv/bin/python


In [2]:
%%writefile src/metrics.py

def iso(double,triple,homerun,ab):
    return (double+2*triple+3*homerun)/ab

def babip (h,hr,ab,so,sf):
    return( h-hr)/(ab-so-hr+sf)

def pythag( rs , ra ,exp=1.83):
    return (rs**exp)/(rs**exp+ra**exp)

Writing src/metrics.py


In [7]:
from src import cpbl,metrics,fetch

data_batter=fetch.fetch_player('batter')
print(data_batter.columns.to_list)

data_batter['iso_cal']=metrics.iso(data_batter['Double'],data_batter['Triple'],data_batter['HR'],data_batter['AB'])
data_batter['babip_cal']=metrics.babip(data_batter['H'],data_batter['HR'],data_batter['AB'],data_batter['SO'],data_batter['SF'])

data_batter_quli=data_batter[data_batter['AB']>200]
iso_bigg=(data_batter_quli['ISO']-data_batter_quli['iso_cal']).abs().max()
print(iso_bigg)
babip_bigg=(data_batter_quli['BABIP']-data_batter_quli['babip_cal']).abs().max()
print(babip_bigg)

<bound method IndexOpsMixin.tolist of Index(['AB', 'AVG', 'BB', 'CS', 'Double', 'GIDP', 'H', 'HBP', 'HR', 'IBB',
       'PA', 'PH', 'R', 'RBI', 'SB', 'SF', 'SH', 'SO', 'Triple', 'games',
       'name', 'abbr', 'BABIP', 'BBK', 'BBp', 'BIPp', 'GBp', 'ISO', 'Kp',
       'OBP', 'OPS', 'OPSplus', 'RC', 'SBp', 'SLG', 'Whiffp', 'wOBA', 'NegWPA',
       'PosWPA', 'RE24', 'WPA'],
      dtype='object')>
0.00048122866894198535
0.0004817518248175223


In [12]:
from src import cpbl , metrics , fetch

game=fetch.fetch_games()
standing=cpbl.rebas_cpbl_half_standing('standard')
home=game.groupby('home').agg(RS=('home_runs','sum'),RA=('away_runs','sum'))
away=game.groupby('away').agg(RS=('away_runs','sum'),RA=('home_runs','sum'))
runs=home.add(away,fill_value=0)
runs['pyt']=metrics.pythag(runs['RS'],runs['RA'])
runs['real_pct']=standing['勝率']
runs['luck']=runs['real_pct']-runs['pyt']
print(runs.sort_values('luck'))

              RS   RA       pyt  real_pct      luck
home                                               
中信兄弟         509  500  0.508161     0.475 -0.033161
味全龍          441  371  0.578423     0.558 -0.020423
統一7-ELEVEn獅  450  402  0.551421     0.538 -0.013421
台鋼雄鷹         456  506  0.452543     0.454  0.001457
富邦悍將         434  458  0.475395     0.492  0.016605
樂天桃猿         390  443  0.441967     0.483  0.041033


In [13]:
!git add .
!git commit -m "Day 6: ISO、BABIP、畢氏勝率"

[main 42c13f2] Day 6: ISO、BABIP、畢氏勝率
 Committer: Leo Chen <leochen@LeodeMacBook-Air.local>
Your name and email address were configured automatically based
on your username and hostname. Please check that they are accurate.
You can suppress this message by setting them explicitly. Run the
following command and follow the instructions in your editor to edit
your configuration file:

    git config --global --edit

After doing this, you may fix the identity used for this commit with:

    git commit --amend --reset-author

 6 files changed, 608 insertions(+), 187 deletions(-)
 create mode 100644 .virtual_documents/D03.ipynb
 create mode 100644 .virtual_documents/D04.ipynb
 create mode 100644 D05.ipynb
 create mode 100644 src/metrics.py


In [21]:
%%writefile -a src/metrics.py

W={'ubb':0.776,'wHBP':0.795,'w1B':0.950,'w2B':1.322,'w3B':1.800,'wHR':2.464}
def woba(hbp,h,double,triple,hr,ab,bb,ibb,sf):
    w=W
    num=(w['ubb']*(bb-ibb)+w['wHBP']*hbp+w['w1B']*(h-double-triple-hr)+w['w2B']*double+w['w3B']*triple+w['wHR']*hr)
    den=ab+bb-ibb+sf+hbp
    return num/den

def woba_scale(lg):
    obp=(lg['H']+lg['BB']+lg['HBP'])/(lg['AB']+lg['BB']+lg['HBP']+lg['SF'])
    woba_real=woba(lg['HBP'],lg['H'],lg['Double'],lg['Triple'],lg['HR'],lg['AB'],lg['BB'],lg['IBB'],lg['SF'])
    return obp/woba_real


Appending to src/metrics.py


In [23]:
from src import fetch,metrics

data=fetch.fetch_player('batter')
b=data.sum(numeric_only=True)
woba_sc=metrics.woba_scale(b)
print(woba_sc)

1.0009590600219966


In [33]:
%%writefile -a src/metrics.py

def fip_const(lg_er,lg_ip,lg_hr,lg_bb,lg_k):
    lg_era=(lg_er*9)/(lg_ip/3)
    return lg_era-((13*lg_hr+3*(lg_bb)-2*lg_k)/(lg_ip/3))

def fip(hr,k,bb,ip,const=2.869):
    return (13*hr+3*bb-2*k)/(ip/3)+const



Appending to src/metrics.py


In [35]:
from src import fetch,metrics

data=fetch.fetch_player('pitcher')
print(data.columns.to_list)
b=data.sum(numeric_only=True)
fip_cons=metrics.fip_const(b['ER'],b['IPOut'],b['HR'],b['BB'],b['SO'])
print(fip_cons)
# data['fip_cal']=metrics.fip(data['HR'],data['SO'],data['BB'],data['IPOut'])-data['FIP']
# print(data)

<bound method IndexOpsMixin.tolist of Index(['BB', 'BF', 'Double', 'ER', 'ERA', 'GIDP', 'H', 'HR', 'IBB', 'IPOut',
       'NP', 'R', 'R_BS', 'R_H', 'R_L', 'R_SV', 'R_W', 'SB', 'SO', 'SP',
       'Triple', 'games', 'name', 'abbr', 'AVG', 'BABIP', 'BB9', 'BBK', 'BBp',
       'ERAplus', 'FIP', 'GBp', 'H9', 'HR9', 'Kp', 'LOBp', 'OBP', 'OPSplus',
       'SBp', 'SLG', 'SO9', 'WHIP', 'Whiffp', 'NegWPA', 'PosWPA', 'RE24',
       'WPA'],
      dtype='object')>
3.0198955613577025
